# Cost Benchmark - Phi-4-Mini
# Measures: Latency, TTFT, GPU Memory, Disk Storage

## Install Packages

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [17]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121
!pip install unsloth

Found existing installation: torch 2.9.0+cu126
Uninstalling torch-2.9.0+cu126:
  Successfully uninstalled torch-2.9.0+cu126
Found existing installation: torchvision 0.24.0+cu126
Uninstalling torchvision-0.24.0+cu126:
  Successfully uninstalled torchvision-0.24.0+cu126
Found existing installation: torchaudio 2.9.0+cu126
Uninstalling torchaudio-2.9.0+cu126:
  Successfully uninstalled torchaudio-2.9.0+cu126
Looking in indexes: https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement torch==2.1.2 (from versions: 2.2.0+cu121, 2.2.1+cu121, 2.2.2+cu121, 2.3.0+cu121, 2.3.1+cu121, 2.4.0+cu121, 2.4.1+cu121, 2.5.0+cu121, 2.5.1+cu121)
ERROR: No matching distribution found for torch==2.1.2
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import unsloth
from unsloth import FastLanguageModel
import torch

print(torch.__version__)  # MUST show 2.1.2

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
2.6.0+cu124


In [2]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_LOGS"] = ""          # optional: keep logs quiet
os.environ["TORCHDYNAMO_VERBOSE"] = "0"

## Import Libraries

In [3]:
import os
import csv
import time
import pandas as pd
# import torch
from datasets import load_dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from tqdm.auto import tqdm
import psutil
import subprocess

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## Mount Drive and Login

In [4]:
drive.mount('/content/drive')

HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in successfully")
else:
    print("No HF_TOKEN found")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Logged in successfully


## Configuration

In [5]:
# Model config
MODEL_NAME = "Phi-4-Mini"
BASE_MODEL = "microsoft/Phi-4-mini-instruct"
LORA_ADAPTER = "Lakshan2003/Phi-4-mini-instruct-customerservice"

# Paths
TEST_DATASET_REPO = "Lakshan2003/slm-cost-benchmark-testset-1000"
RESULTS_DIR = f"/content/drive/MyDrive/fyp-2025/cost_benchmark_results/Phi-4-Mini"
RESULTS_CSV = os.path.join(RESULTS_DIR, f"Phi-4-Mini_results.csv")
METRICS_CSV = os.path.join(RESULTS_DIR, f"Phi-4-Mini_metrics.csv")
OUTPUT_REPO = f"Lakshan2003/Phi-4-Mini-cost-benchmark-results"

os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Model: Phi-4-Mini")
print(f"Results will be saved to: {RESULTS_DIR}")

Model: Phi-4-Mini
Results will be saved to: /content/drive/MyDrive/fyp-2025/cost_benchmark_results/Phi-4-Mini


## Load Test Dataset

In [6]:
test_dataset = load_dataset(TEST_DATASET_REPO, split="train")
print(f"Loaded {len(test_dataset)} test samples")

Loaded 1000 test samples


## GPU Memory Tracking Functions

In [7]:
def get_gpu_memory_mb():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024 ** 2)
    return 0

def get_model_disk_size_gb(model_path):
    """Calculate disk storage for model files in GB"""
    try:
        result = subprocess.run(
            ["du", "-sb", model_path],
            capture_output=True,
            text=True
        )
        size_bytes = int(result.stdout.split()[0])
        return size_bytes / (1024 ** 3)
    except:
        return 0

## Load Model

In [8]:
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

use_bf16 = torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else None

# Load base model + processor with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "microsoft/Phi-4-mini-instruct",  # Base microsoft/Phi-4-mini-instruct
    max_seq_length = 512,
    dtype = torch.float16,
    device_map="auto",
    load_in_4bit=not use_bf16  # fallback to 4-bit when no BF16
)


#  LoRA adapter
adapter_path = "Lakshan2003/Phi-4-mini-instruct-customerservice"
model = PeftModel.from_pretrained(model, adapter_path)

# Merge the adapter
model = model.merge_and_unload()

model.eval()

print("Model loaded successfully")

# Get model disk size
cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
MODEL_DISK_SIZE_GB = get_model_disk_size_gb(cache_dir)
print(f"Model disk storage: {MODEL_DISK_SIZE_GB:.2f} GB")

/tmp/ipython-input-2561609896.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Phi3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully
Model disk storage: 7.20 GB


## Prompt Template

In [9]:
# Chat template for Phi-4-Mini
PROMPT_TEMPLATE = """<|system|>{instruction}<|end|>
<|user|>Summarized Conversation History:
{history}

Client Question:
{client_question}<|end|>
<|assistant|>"""

# Generation config
GENERATION_CONFIG = {
    "max_new_tokens": 128,
    "temperature": 0.7,
    "do_sample": True,
    "top_p": 0.9,
    "top_k": 50,
}

def build_prompt(example):
    """Build prompt from example"""
    return PROMPT_TEMPLATE.format(
        instruction=example.get("instruction", ""),
        history=example.get("history_summary", ""),
        client_question=example.get("client_question", "")
    )

## Resume Logic

In [10]:
# Track processed IDs
processed_ids = set()

if os.path.exists(RESULTS_CSV):
    try:
        prev_df = pd.read_csv(RESULTS_CSV)
        if "conversation_id" in prev_df.columns:
            processed_ids = set(prev_df["conversation_id"].astype(str))
            print(f"Resuming from {len(processed_ids)} saved rows")
    except Exception as e:
        print(f"CSV unreadable, starting fresh: {e}")

# Filter unprocessed samples
to_run = []
for ex in test_dataset:
    cid = str(ex.get("conversation_id", ""))
    if cid and cid not in processed_ids:
        to_run.append(ex)

print(f"Pending samples: {len(to_run)}")

Pending samples: 1000


## Inference Loop with Metrics

In [11]:
# Setup
torch.set_grad_enabled(False)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "left"

# EOS token IDs
eos_ids = [tokenizer.eos_token_id]
end_token = "<|im_end|>"
if hasattr(tokenizer, "get_vocab") and end_token in tokenizer.get_vocab():
    end_id = tokenizer.convert_tokens_to_ids(end_token)
    if end_id != tokenizer.eos_token_id:
        eos_ids = [tokenizer.eos_token_id, end_id]

RESULT_COLS = [
    "conversation_id",
    "instruction",
    "history",
    "history_summary",
    "client_question",
    "ground_truth",
    "generated_answer",
]

METRIC_COLS = [
    "conversation_id",
    "latency_seconds",
    "ttft_seconds",
    "gpu_memory_mb",
    "disk_storage_gb",
]

pbar = tqdm(total=len(to_run), desc="Generating", unit="sample")

for ex in to_run:
    conversation_id = str(ex.get("conversation_id", ""))
    instruction = ex.get("instruction", "")
    history = ex.get("history", "")
    history_summary = ex.get("history_summary", "")
    client_question = ex.get("client_question", "")
    ground_truth = ex.get("ground_truth", "")

    input_prompt = build_prompt(ex)

    # TOKENIZATION
    enc = tokenizer(
        [input_prompt],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=tokenizer.model_max_length,
    ).to(model.device)

    torch.cuda.reset_peak_memory_stats()
    gpu_mem_before = get_gpu_memory_mb()

    # GENERATE FIRST TOKEN (TTFT)
    torch.cuda.synchronize()
    start_ttft = time.time()

    with torch.no_grad():
        first_token_out = model.generate(
            **enc,
            max_new_tokens=1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=eos_ids,
            return_dict_in_generate=True,
            output_scores=False,
            use_cache=True,
        )

    torch.cuda.synchronize()
    ttft = time.time() - start_ttft

    # GENERATE REMAINING TOKENS (FULL LATENCY)
    torch.cuda.synchronize()
    start_latency = time.time()

    with torch.no_grad():
        full_out = model.generate(
            **enc,
            **GENERATION_CONFIG,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=eos_ids,
            return_dict_in_generate=True,
            use_cache=True,
        )

    torch.cuda.synchronize()
    latency = time.time() - start_latency

    # MEMORY
    gpu_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    # DECODE
    seqs = full_out.sequences
    start_idx = enc.input_ids.shape[1]
    generated_text = tokenizer.decode(
        seqs[0, start_idx:],
        skip_special_tokens=True
    ).strip()

    # SAVE RESULTS
    result_row = {
        "conversation_id": conversation_id,
        "instruction": instruction,
        "history": history,
        "history_summary": history_summary,
        "client_question": client_question,
        "ground_truth": ground_truth,
        "generated_answer": generated_text,
    }

    metrics_row = {
        "conversation_id": conversation_id,
        "latency_seconds": latency,
        "ttft_seconds": ttft,
        "gpu_memory_mb": gpu_memory_mb,
        "disk_storage_gb": MODEL_DISK_SIZE_GB,
    }

    pd.DataFrame([result_row], columns=RESULT_COLS).to_csv(
        RESULTS_CSV,
        mode="a",
        header=not os.path.exists(RESULTS_CSV),
        index=False,
        quoting=csv.QUOTE_MINIMAL,
    )

    pd.DataFrame([metrics_row], columns=METRIC_COLS).to_csv(
        METRICS_CSV,
        mode="a",
        header=not os.path.exists(METRICS_CSV),
        index=False,
        quoting=csv.QUOTE_MINIMAL,
    )

    pbar.update(1)

pbar.close()
print("\nInference completed!")

Generating:   0%|          | 0/1000 [00:00<?, ?sample/s]


Inference completed!


## Calculate Summary Statistics

In [12]:
# Load metrics
metrics_df = pd.read_csv(METRICS_CSV)

print("\n=== Summary Statistics ===")
print(f"\nTotal Samples: {len(metrics_df)}")
print(f"\nLatency (seconds):")
print(f"  Mean: {metrics_df['latency_seconds'].mean():.4f}")
print(f"  Median: {metrics_df['latency_seconds'].median():.4f}")
print(f"  Std: {metrics_df['latency_seconds'].std():.4f}")
print(f"  Min: {metrics_df['latency_seconds'].min():.4f}")
print(f"  Max: {metrics_df['latency_seconds'].max():.4f}")

print(f"\nTTFT (seconds):")
print(f"  Mean: {metrics_df['ttft_seconds'].mean():.4f}")
print(f"  Median: {metrics_df['ttft_seconds'].median():.4f}")
print(f"  Std: {metrics_df['ttft_seconds'].std():.4f}")

print(f"\nGPU Memory (MB):")
print(f"  Mean: {metrics_df['gpu_memory_mb'].mean():.2f}")
print(f"  Max: {metrics_df['gpu_memory_mb'].max():.2f}")

print(f"\nDisk Storage (GB): {MODEL_DISK_SIZE_GB:.2f}")

# Save summary
summary_path = os.path.join(RESULTS_DIR, f"Phi-4-Mini_summary.txt")
with open(summary_path, "w") as f:
    f.write(f"Model: Phi-4-Mini\n")
    f.write(f"Total Samples: {len(metrics_df)}\n\n")
    f.write(f"Latency (seconds):\n")
    f.write(f"  Mean: {metrics_df['latency_seconds'].mean():.4f}\n")
    f.write(f"  Median: {metrics_df['latency_seconds'].median():.4f}\n")
    f.write(f"  Std: {metrics_df['latency_seconds'].std():.4f}\n")
    f.write(f"\nTTFT (seconds):\n")
    f.write(f"  Mean: {metrics_df['ttft_seconds'].mean():.4f}\n")
    f.write(f"  Median: {metrics_df['ttft_seconds'].median():.4f}\n")
    f.write(f"\nGPU Memory (MB):\n")
    f.write(f"  Mean: {metrics_df['gpu_memory_mb'].mean():.2f}\n")
    f.write(f"  Max: {metrics_df['gpu_memory_mb'].max():.2f}\n")
    f.write(f"\nDisk Storage (GB): {MODEL_DISK_SIZE_GB:.2f}\n")

print(f"\nSummary saved to: {summary_path}")


=== Summary Statistics ===

Total Samples: 1000

Latency (seconds):
  Mean: 2.0749
  Median: 1.9825
  Std: 0.6368
  Min: 0.8505
  Max: 5.2480

TTFT (seconds):
  Mean: 0.0570
  Median: 0.0560
  Std: 0.0131

GPU Memory (MB):
  Mean: 7479.48
  Max: 7515.92

Disk Storage (GB): 7.20

Summary saved to: /content/drive/MyDrive/fyp-2025/cost_benchmark_results/Phi-4-Mini/Phi-4-Mini_summary.txt


## Push to HuggingFace

In [13]:
from datasets import Dataset

# Load both CSVs
results_df = pd.read_csv(RESULTS_CSV)
metrics_df = pd.read_csv(METRICS_CSV)

# Merge on conversation_id
final_df = results_df.merge(metrics_df, on="conversation_id", how="inner")

# Save combined version locally
FINAL_CSV = os.path.join(RESULTS_DIR, "combined_results_metrics.csv")
final_df.to_csv(FINAL_CSV, index=False)

print(f"Combined file saved: {FINAL_CSV}")

Combined file saved: /content/drive/MyDrive/fyp-2025/cost_benchmark_results/Phi-4-Mini/combined_results_metrics.csv


In [14]:
final_dataset = Dataset.from_pandas(final_df)

final_dataset.push_to_hub(OUTPUT_REPO, private=True)

print(f"Pushed combined dataset to: {OUTPUT_REPO}")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  46%|####6     |  530kB / 1.15MB            

README.md:   0%|          | 0.00/710 [00:00<?, ?B/s]

Pushed combined dataset to: Lakshan2003/Phi-4-Mini-cost-benchmark-results
